In [6]:
!pip install torch

In [7]:
from huggingface_hub import notebook_login
notebook_login()

In [8]:
!pip install peft

In [9]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM

config = PeftConfig.from_pretrained("grounded-ai/phi3-hallucination-judge")
base_model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
model = PeftModel.from_pretrained(base_model, "grounded-ai/phi3-hallucination-judge")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [10]:
!pip install transformers

In [11]:
!pip install torch

In [12]:
!pip install tokenizers==0.19.1

  Using cached tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.6 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.19.1 which is incompatible.


In [ ]:
# Import Hugging Face helpers
# AutoModelForCausalLM  -> loads a text-generation (causal language) model
# AutoTokenizer        -> converts text into tokens the model understands
# pipeline             -> high-level wrapper that runs tokenizer + model + decoding
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# PyTorch is used to run the model on CPU or GPU
import torch

# Low-level tokenizer library (NOT used in this script, can be removed safely)
import tokenizers


# ---------------------------------------------------------
# Load tokenizer for the Phi-3 instruct model
# Tokenizer:
# - splits text into tokens
# - adds special tokens required by the model
# ---------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct"
)


# ---------------------------------------------------------
# Function to build the evaluation prompt
# This prompt tells the model to act as a "hallucination judge"
# ---------------------------------------------------------
def format_input(reference, query, response):
    prompt = f"""Your job is to evaluate whether a machine learning model has hallucinated or not.
    A hallucination occurs when the response is coherent but factually incorrect or nonsensical
    outputs that are not grounded in the provided context.
    You are given the following information:
    ####INFO####
    [Knowledge]: Walrus are the largest mammal
    [User Input]: What is the smallest mammal?
    [Model Response]: Walrus
    ####END INFO####
    Based on the information provided is the model output a hallucination? Respond with only "yes" or "no"
    """
    return prompt


# ---------------------------------------------------------
# Build the final text prompt
# (parameters passed here are ignored in current implementation)
# ---------------------------------------------------------
text = format_input(
    query='',
    response='',
    reference=''
)


# ---------------------------------------------------------
# Wrap the prompt in chat-style format
# Many instruct models expect role-based messages
# ---------------------------------------------------------
messages = [
    {
        "role": "user",     # role of the speaker
        "content": text     # actual prompt text
    }
]


# ---------------------------------------------------------
# Create a text-generation pipeline
# pipeline does:
# 1. Tokenization
# 2. Model inference
# 3. Decoding tokens back to text
# ---------------------------------------------------------
pipe = pipeline(
    "text-generation",     # task type
    model=base_model,      # language model (must be defined earlier)
    tokenizer=tokenizer,  # tokenizer for that model
)


# ---------------------------------------------------------
# Generation configuration
# Controls how the model generates text
# ---------------------------------------------------------

generation_args = {
      "max_new_tokens": 2,       # generate at most 2 new tokens ("yes" / "no")
      "return_full_text": False, # return only generated output, not prompt
      "temperature": 0.01,       # very low randomness (deterministic)
      "do_sample": True,         # enable sampling (can be False for judges)
  }
          
# ---------------------------------------------------------
# Run the pipeline
# This executes:
# tokenizer -> model.generate -> decode
# ---------------------------------------------------------
output = pipe(messages, **generation_args)


# ---------------------------------------------------------
# Print the result
# output is a LIST of dictionaries
# Each dict contains "generated_text"
# ---------------------------------------------------------
print(f'Hallucination: {output[0]}')




Device set to use cpu


Hallucination: {'generated_text': ' yes'}
